# 🧩 MTP 重みの実体 × ローダ期待キー

`mtp.*` キーと vLLM の読み取り規約が一致するか。


In [ ]:
import glob
import pathlib
import subprocess
import sys
import zipfile

subprocess.run([sys.executable, "-m", "pip", "download", "--no-cache-dir",
                "--no-deps", "vllm==0.32.0", "-d", "/tmp/w"],
               capture_output=True, text=True)
whl = glob.glob("/tmp/w/*.whl")
print("wheel:", whl)
if not whl:
    raise SystemExit("wheel 取得失敗")
zipfile.ZipFile(whl[0]).extractall("/tmp/vl")
root = pathlib.Path("/tmp/vl/vllm")
print("py files:", len(list(root.rglob("*.py"))))

import json
import os
import struct


def find_root():
    import glob
    slug = "qwen38-27b-uncensored-hauhaucs-awq-mtp"
    ordered = []
    for pat in ("/kaggle/input/*/config.json", "/kaggle/input/*/*/config.json",
                "/kaggle/input/*/*/*/config.json"):
        ordered += glob.glob(pat)
    # slug 一致を最優先。無ければ qwen を含むもの、最後に全候補。
    cands = ([c for c in ordered if slug in c]
             or [c for c in ordered if "qwen" in c.lower()]
             or ordered)
    print("config.json 候補:", cands[:3])
    if not cands:
        print("/kaggle/input 直下:", glob.glob("/kaggle/input/*")[:10])
        raise SystemExit("dataset が見つからない")
    return cands[0].rsplit("/", 1)[0]


ROOT = find_root()



def header(path):
    with open(path, "rb") as f:
        n = struct.unpack("<Q", f.read(8))[0]
        return json.loads(f.read(n).decode("utf-8", "replace"))


print("=== Dataset 内のファイル ===")
for name in sorted(os.listdir(ROOT)):
    p = os.path.join(ROOT, name)
    size = "DIR" if os.path.isdir(p) else os.path.getsize(p)
    print(f"  {name:32s} {size}")

print("\n=== safetensors ヘッダ ===")
allkeys = {}
for name in ("model.safetensors", "model-mtp.safetensors"):
    p = os.path.join(ROOT, name)
    if not os.path.isfile(p):
        print("  missing:", name)
        continue
    h = header(p)
    meta = h.pop("__metadata__", None)
    total = sum(v["data_offsets"][1] - v["data_offsets"][0] for v in h.values())
    print(f"  {name}: tensors={len(h)} bytes={total / 2**30:.2f}GiB meta={meta}")
    dts = {}
    for v in h.values():
        dts[v["dtype"]] = dts.get(v["dtype"], 0) + 1
    print("    dtypes:", dts)
    ks = sorted(h)
    allkeys[name] = ks
    print("    first 6:", ks[:6])
    pref = {}
    for k in ks:
        parts = k.split(".")
        p3 = ".".join(parts[:3])
        pref[p3] = pref.get(p3, 0) + 1
    top = sorted(pref.items(), key=lambda x: -x[1])[:12]
    print("    prefixes:", top)
    packed = [k for k in ks if "packed" in k][:4]
    print("    weight_packed 例:", packed)
    print("    scale 例:", [k for k in ks if k.endswith("weight_scale")][:3])

print("\n=== index.json ===")
ip = os.path.join(ROOT, "model.safetensors.index.json")
if os.path.isfile(ip):
    idx = json.load(open(ip))
    wm = idx.get("weight_map", {})
    print("  entries:", len(wm), "| files:", sorted(set(wm.values())))
    mtp_keys = [k for k in wm if "mtp" in k.lower()]
    print("  mtp keys in index:", len(mtp_keys), mtp_keys[:6])
else:
    print("  index.json なし（単一シャード）")

print("\n=== vLLM が MTP をどう読むか ===")
cand = [f for f in root.rglob("*.py")
        if "mtp" in f.name.lower() or "spec_decode" in str(f)]
for f in sorted(cand)[:12]:
    t = f.read_text(errors="replace")
    lines = t.splitlines()
    hits = [i for i, l in enumerate(lines) if "mtp" in l.lower()]
    if not hits:
        continue
    print(f"--- {f.relative_to(root)} ({len(hits)} hits) ---")
    for i in hits[:10]:
        print(f"  {i + 1}: {lines[i].strip()[:150]}")

qf = root / "model_executor/models/qwen3_5_mtp.py"
if qf.exists():
    print("\n=== qwen3_5_mtp.py の重み名 ===")
    t = qf.read_text(errors="replace")
    for i, l in enumerate(t.splitlines(), 1):
        s = l.strip()
        if any(k in s for k in ("weight", "fc", "norm", "prefix", "load",
                                "class ", "def ")):
            print(f"  {i}: {s[:150]}")

print("\n=== 実キーと期待プレフィックスの照合 ===")
actual = allkeys.get("model-mtp.safetensors", [])
print("actual sample:", actual[:10])
for want in ("mtp.", "model.mtp.", "model.layers.64.", "language_model.mtp."):
    hit = [k for k in actual if k.startswith(want)]
    print(f"  {want:22s} -> {len(hit)}")
